# Fake Job Posting Detection — Leakage-Safe Baseline

This is a **new, review-oriented baseline** based on the original master's project. It is not a reproduction of the original notebook's reported results. The included dataset is a stratified *demonstration sample* and is not representative of the full dataset's prevalence. No evaluation results are claimed until the notebook is executed.

The original student notebooks are retained separately under `original_notebooks/`.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score

path = Path("data/fake_job_postings_sample.csv")
df = pd.read_csv(path)
print("Rows:", len(df), "Fraud rate:", round(df.fraudulent.mean(), 4))

In [ ]:
text_columns = ["title", "company_profile", "description", "requirements", "benefits"]
X = df[text_columns].fillna("").astype(str).agg(" ".join, axis=1)
y = df["fraudulent"].astype(int)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)
print("Train / validation / test:", len(X_train), len(X_val), len(X_test))

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
model.fit(X_train, y_train)
val_scores = model.predict_proba(X_val)[:, 1]
print("Validation PR-AUC:", average_precision_score(y_val, val_scores))

In [ ]:
# Choose a threshold on validation data only, optimizing fraud-class F1.
from sklearn.metrics import f1_score
import numpy as np
thresholds = np.linspace(0.1, 0.9, 17)
threshold = max(thresholds, key=lambda t: f1_score(y_val, (val_scores >= t).astype(int), zero_division=0))
print("Validation-selected threshold:", round(float(threshold), 3))
test_scores = model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= threshold).astype(int)
print(classification_report(y_test, test_predictions, digits=3, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_predictions))
print("Test PR-AUC:", average_precision_score(y_test, test_scores))
print("Test ROC-AUC:", roc_auc_score(y_test, test_scores))

## Methodological notes
- TF-IDF is fitted **only on the training set** via the pipeline.
- The decision threshold is selected on validation data, never on test data.
- The test set is held out until final evaluation.
- The demonstration sample is artificially enriched with fraudulent listings; **do not extrapolate its performance to real-world prevalence**.
- Potential near-duplicate postings and dataset provenance should be investigated before claiming deployable performance.
- This baseline does not reproduce the original XGBoost or LSTM experiments.